In [356]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.3f}'.format)

# Helper Functions

## Importante

In [357]:
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from utils.utils import load_from_pickle
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
import statsmodels.api as sm

validated_testset = load_from_pickle("validated_testset")
non_a_qa_pairs = validated_testset[validated_testset.validate != "A"]
non_a_questions = non_a_qa_pairs["cleaned_question"].tolist()
non_a_questions


def get_metrics_df(data):
    df = pd.DataFrame.from_dict(data, orient='index')
    
    filtered_data = {k: {metric: v for metric, v in metrics.items() 
                     if metric not in ['ground_truth', 'output', 'RAGAS_Answer_Similarity']} 
                 for k, metrics in data.items()}
    df = pd.DataFrame.from_dict(filtered_data, orient='index')

    return df

def filter_metrics(metrics):
    for k, v in metrics.items():
        questions = v['question'].to_list()
        mask = ~pd.Series(questions).isin(non_a_questions)
        metrics[k] = metrics[k][mask]
        metrics[k] = metrics[k].drop(['question'], axis=1)

    return metrics

def check_normality(metrics):
    # Run Shapiro-Wilk test for normality
    normality_results = {}
    for column in metrics.columns:
        stat, p_value = stats.shapiro(metrics[column])
        normality_results[column] = {
            'Shapiro-Wilk Statistic': stat, 
            'p-value': p_value,
        }

    normality_df = pd.DataFrame(normality_results).T
    print("Shapiro-Wilk Test Results:")
    print(normality_df)

    # Visualize normality with histograms and Q-Q plots
    for column in metrics.columns:
        plt.figure(figsize=(10, 4))

        # Histogram with KDE
        plt.subplot(1, 2, 1)
        sns.histplot(metrics[column], kde=True)
        plt.title(f'Histogram of {column}')

        # Q-Q plot
        plt.subplot(1, 2, 2)
        stats.probplot(metrics[column], dist="norm", plot=plt)
        plt.title(f'Q-Q Plot of {column}')

        plt.tight_layout()
        plt.show()
    
    return normality_df

def t_test(metrics1, metrics2):
    t_test_results = {}
    mean_diffs = metrics1.mean() - metrics2.mean()
    
    for column in metrics2.columns:
        t_stat, p_value = stats.ttest_rel(metrics1[column], metrics2[column])
        t_test_results[column] = {
            't-statistic': t_stat,
            'p-value': p_value,
            'significant': p_value <= 0.05,
            'better_results': 'first metric' if mean_diffs[column] > 0 else 'second metric',
            'mean_difference': mean_diffs[column],
        }
    
    t_test_df = pd.DataFrame(t_test_results).T
    t_test_df["significant"] = t_test_df["significant"].astype(bool)
    t_test_df["better_results"] = t_test_df["better_results"].astype(str)
    
    sig_count = t_test_df['significant'].sum()
    first_count = (t_test_df['better_results'] == 'first metric').sum()
    second_count = (t_test_df['better_results'] == 'second metric').sum()
    
    print(f"Significant on {sig_count} out of {len(t_test_df)} --> {sig_count / len(t_test_df) * 100:.2f}%")
    print(f"First metric preferred in {first_count} metrics ; second in {second_count} metrics")
    return t_test_df

def wilcoxon_test(metrics1, metrics2):
    wilcoxon_results = {}
    for column in metrics2.columns:
        stat, p_value = stats.wilcoxon(metrics2[column], metrics1[column])
        wilcoxon_results[column] = {'Wilcoxon Statistic': stat, 'p-value': p_value}
    wilcoxon_df = pd.DataFrame(wilcoxon_results).T
    wilcoxon_df["significant"] = wilcoxon_df["p-value"] <= 0.05
    return wilcoxon_df

def anova_test(*metrics):
    anova_results = {}
    
    metric_names = [metric[0] for metric in metrics]
    metrics_data = [metric[1] for metric in metrics]
    
    sig_count = 0
    best_metric_count = {name: 0 for name in metric_names}
    
    for column in metrics_data[0].columns:
        metric_data_column = [metric[column] for metric in metrics_data]
        
        f_stat, p_value = stats.f_oneway(*metric_data_column)
        
        mean_values = [metric[column].mean() for metric in metrics_data]
        best_metric_idx = mean_values.index(max(mean_values))
        
        if p_value <= 0.05:
            sig_count += 1
        
        best_metric_count[metric_names[best_metric_idx]] += 1
        
        anova_results[column] = {
            'F-statistic': f_stat,
            'p-value': p_value,
            'significant': p_value <= 0.05, 
            'best_metric': metric_names[best_metric_idx],
            'mean_difference': max(mean_values) - min(mean_values),
            # 'mean_values': mean_values  
        }
    
    anova_df = pd.DataFrame(anova_results).T
    anova_df["significant"] = anova_df["significant"].astype(bool)
    
    sig_percent = sig_count / len(anova_results) * 100
    print(f"Significant on {sig_count} out of {len(anova_results)} --> {sig_percent:.2f}%")
    for metric_name, count in best_metric_count.items():
        print(f"{metric_name} was the best in {count} metrics")

    tukey_results = {}
    for column in metrics_data[0].columns:
        if anova_results[column]['significant']:
            all_data = []
            group_labels = []
            for metric_idx, metric_name in enumerate(metric_names):
                all_data.extend(metrics_data[metric_idx][column].values)
                group_labels.extend([metric_name] * len(metrics_data[metric_idx][column]))
            
            # Convert to pandas Series for Tukey HSD
            data = pd.Series(all_data)
            groups = pd.Series(group_labels)

            tukey = pairwise_tukeyhsd(endog=data, groups=groups, alpha=0.05)
            tukey_summary = tukey.summary()
            tukey_results[column] = tukey_summary
    
    print("\nTukey's HSD Results:")
    for column, result in tukey_results.items():
        print(f"\nFor column: {column}")
        print(result)
    
    return anova_df

## Experimentale

In [358]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

def three_way_anova_test(data, dependent_var, factor1, factor2, factor3):
    anova_results = {}

    # Build the formula for the three-way ANOVA
    formula = f"{dependent_var} ~ C({factor1}) + C({factor2}) + C({factor3}) + " \
              f"C({factor1}):C({factor2}) + C({factor1}):C({factor3}) + " \
              f"C({factor2}):C({factor3}) + C({factor1}):C({factor2}):C({factor3})"

    # Fit the model
    model = ols(formula, data=data).fit()

    # Perform ANOVA
    anova_table = sm.stats.anova_lm(model, typ=2)

    # Print and return the results
    print(anova_table)
    return anova_table

# Analysis

## One-Way ANOVA

In [359]:
from utils.utils import load_from_pickle

directories = [
    "baseline",
    "cos_512_128_bottom", "cos_512_128_alternate", "cos_512_128_top", "cos_512_128_nocontext", "cos_256_64_top", "cos_1024_256_top", 
    "sci_512_128_nonoise", "sci_512_128_single", "sci_512_128_multiple", "sci_256_64_nonoise", "sci_1024_256_nonoise", 
    "sp_zero", "sp_few", "sp_cot", 
    "llmtemp_low", "llmtemp_mid", "llmtemp_high", 
    "llmtopp_low", "llmtopp_mid", "llmtopp_high", 
    "llmmodel_llama3b",
    "sptp_zll", "sptp_zml", "sptp_zhl", "sptp_zlm", "sptp_zlh", "sptp_zmm", "sptp_zmh", "sptp_zhm", "sptp_zhh",
    "sptp_fll", "sptp_fml", "sptp_fhl", "sptp_flm", "sptp_flh", "sptp_fmm", "sptp_fmh", "sptp_fhm", "sptp_fhh", 
    "sptp_cll", "sptp_cml", "sptp_chl", "sptp_clm", "sptp_clh", "sptp_cmm", "sptp_cmh", "sptp_chm", "sptp_chh",
    "cs_nc", "cs_nn", "cs_bs", "cs_bm", "cs_as", "cs_am", "cs_ts", "cs_tm"
]

metrics = {}

for dir_name in directories:
    path = f"llm_experiments/{dir_name}"
    data = load_from_pickle(path)
    metrics[dir_name] = get_metrics_df(data)

metrics = filter_metrics(metrics)

In [360]:
len(directories)

57

In [377]:
dirs = [
    "cs_nc", "cs_nn", "cs_bs", "cs_bm", "cs_as", "cs_am", "cs_ts", "cs_tm"
]

metrics_avg = {}

for di in dirs:
    avg_metrics = metrics[di].mean()
    print(di)
    # print(avg_metrics)

    for i, avg_metric in enumerate(avg_metrics):
        print(avg_metric)
        if i in metrics_avg:
            metrics_avg[i] += avg_metric
        else:
            metrics_avg[i] = avg_metric
        
    
    print('-'*40)


cs_nc
0.1485876438551516
0.2436438907199252
0.6304611843850635
0.3291677961898779
0.1563223786745105
0.423954285865141
0.21492394388299138
0.20178349811659146
0.5392927225323417
0.27549773821929896
0.39211701782447406
0.8122796669448303
0.8787921010101037
0.8438481329410572
----------------------------------------
cs_nn
0.2510193727302809
0.3463080877873652
0.6764442731475931
0.43429443751788344
0.2645628087954197
0.5217587108560878
0.33324620772913116
0.30815926529512566
0.6065768831079312
0.3875837935861741
0.49677845087866707
0.8171364170114276
0.9054147784895296
0.8586312431295636
----------------------------------------
cs_bs
0.2597706846356541
0.3636823663868444
0.6290635652356541
0.4372082531979214
0.2798420097260014
0.4848871757121061
0.3375873341170386
0.3280658644299154
0.570706360401919
0.39544585656973363
0.47038815383777105
0.8130223755633772
0.905533188949002
0.8563785672826997
----------------------------------------
cs_bm
0.25796895479843507
0.36246836252781023
0.623701

In [382]:
for k, v in metrics_avg.items():
    print(f"{v/len(dirs):.3f}")

0.245
0.349
0.634
0.424
0.264
0.482
0.323
0.312
0.571
0.381
0.463
0.814
0.902
0.855


In [362]:
# normality_df = check_normality(metrics)

In [363]:
# t_test_df = t_test(metrics['cs_tm'], metrics['cs_ts'])
# print(t_test_df)

In [367]:
cos_metrics = [
    ('cos_512_128_bottom', metrics['cos_512_128_bottom']),
    ('cos_512_128_alternate', metrics['cos_512_128_alternate']),
    ('cos_512_128_top', metrics['cos_512_128_top']),
    ('cos_512_128_nocontext', metrics['cos_512_128_nocontext']),
    # ('cos_256_64_top', metrics['cos_256_64_top']),
    # ('cos_1024_256_top', metrics['cos_1024_256_top']),
]

sci_metrics = [
    ('sci_512_128_nonoise', metrics['sci_512_128_nonoise']),
    ('sci_512_128_single', metrics['sci_512_128_single']),
    ('sci_512_128_multiple', metrics['sci_512_128_multiple']),
    # ('sci_256_64_nonoise', metrics['sci_256_64_nonoise']),
    # ('sci_1024_256_nonoise', metrics['sci_1024_256_nonoise']),
]

sp_metrics = [
    ('sp_zero', metrics['sp_zero']),
    ('sp_few', metrics['sp_few']),
    ('sp_cot', metrics['sp_cot']),
]

llmtemp_metrics = [
    ('llmtemp_low', metrics['llmtemp_low']),
    ('llmtemp_mid', metrics['llmtemp_mid']),
    ('llmtemp_high', metrics['llmtemp_high']),
]

llmtopp_metrics = [
    ('llmtopp_low', metrics['llmtopp_low']),
    ('llmtopp_mid', metrics['llmtopp_mid']),
    ('llmtopp_high', metrics['llmtopp_high']),
]

llmmodel_metrics = [
    ('baseline', metrics['baseline']),
    ('llmmodel_llama3b', metrics['llmmodel_llama3b']),
]

sptp_metrics = [
    ('sptp_zll', metrics['sptp_zll']),
    ('sptp_zml', metrics['sptp_zml']),
    ('sptp_zhl', metrics['sptp_zhl']),
    ('sptp_zlm', metrics['sptp_zlm']),
    ('sptp_zlh', metrics['sptp_zlh']),
    ('sptp_zmm', metrics['sptp_zmm']),
    ('sptp_zmh', metrics['sptp_zmh']),
    ('sptp_zhm', metrics['sptp_zhm']),
    ('sptp_zhh', metrics['sptp_zhh']),
    ('sptp_fll', metrics['sptp_fll']),
    ('sptp_fml', metrics['sptp_fml']),
    ('sptp_fhl', metrics['sptp_fhl']),
    ('sptp_flm', metrics['sptp_flm']),
    ('sptp_flh', metrics['sptp_flh']),
    ('sptp_fmm', metrics['sptp_fmm']),
    ('sptp_fmh', metrics['sptp_fmh']),
    ('sptp_fhm', metrics['sptp_fhm']),
    ('sptp_fhh', metrics['sptp_fhh']),
    ('sptp_cll', metrics['sptp_cll']),
    ('sptp_cml', metrics['sptp_cml']),
    ('sptp_chl', metrics['sptp_chl']),
    ('sptp_clm', metrics['sptp_clm']),
    ('sptp_clh', metrics['sptp_clh']),
    ('sptp_cmm', metrics['sptp_cmm']),
    ('sptp_cmh', metrics['sptp_cmh']),
    ('sptp_chm', metrics['sptp_chm']),
    ('sptp_chh', metrics['sptp_chh']),
]

cs_metrics = [
    ('cs_nc', metrics['cs_nc']),
    ('cs_nn', metrics['cs_nn']),
    ('cs_bs', metrics['cs_bs']),
    ('cs_bm', metrics['cs_bm']),
    ('cs_as', metrics['cs_as']),
    ('cs_am', metrics['cs_am']),
    ('cs_ts', metrics['cs_ts']),
    ('cs_tm', metrics['cs_tm']),
]

anova_df = anova_test(*cs_metrics)
print(anova_df)

Significant on 14 out of 14 --> 100.00%
cs_nc was the best in 0 metrics
cs_nn was the best in 6 metrics
cs_bs was the best in 0 metrics
cs_bm was the best in 0 metrics
cs_as was the best in 0 metrics
cs_am was the best in 0 metrics
cs_ts was the best in 6 metrics
cs_tm was the best in 2 metrics


/raid/p-storage/joshuamihai_e64ef832/.local/lib/python3.10/site-packages/scipy/integrate/_quadpack_py.py:1272: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  quad_r = quad(f, low, high, args=args, full_output=self.full_output,



Tukey's HSD Results:

For column: BLEU
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
 cs_am  cs_as   0.0037 0.9965 -0.0121  0.0196  False
 cs_am  cs_bm   0.0005    1.0 -0.0154  0.0163  False
 cs_am  cs_bs   0.0023 0.9999 -0.0136  0.0181  False
 cs_am  cs_nc  -0.1089    0.0 -0.1248 -0.0931   True
 cs_am  cs_nn  -0.0065 0.9193 -0.0223  0.0093  False
 cs_am  cs_tm   0.0042  0.993 -0.0116    0.02  False
 cs_am  cs_ts   0.0058 0.9534   -0.01  0.0217  False
 cs_as  cs_bm  -0.0033 0.9985 -0.0191  0.0126  False
 cs_as  cs_bs  -0.0015    1.0 -0.0173  0.0144  False
 cs_as  cs_nc  -0.1127    0.0 -0.1285 -0.0968   True
 cs_as  cs_nn  -0.0102   0.51 -0.0261  0.0056  False
 cs_as  cs_tm   0.0004    1.0 -0.0154  0.0163  False
 cs_as  cs_ts   0.0021 0.9999 -0.0137  0.0179  False
 cs_bm  cs_bs   0.0018    1.0  -0.014  0.0176  False
 cs_bm  cs_nc  -0.1094    0.0 -0.1252 -0.0935   True
 cs_bm

## Three Way ANOVA

In [352]:
dirs = [
    "sptp_zll", "sptp_zml", "sptp_zhl", "sptp_zlm", "sptp_zlh", "sptp_zmm", "sptp_zmh", "sptp_zhm", "sptp_zhh",
    "sptp_fll", "sptp_fml", "sptp_fhl", "sptp_flm", "sptp_flh", "sptp_fmm", "sptp_fmh", "sptp_fhm", "sptp_fhh", 
    "sptp_cll", "sptp_cml", "sptp_chl", "sptp_clm", "sptp_clh", "sptp_cmm", "sptp_cmh", "sptp_chm", "sptp_chh",
]

data_list = []
for name, df in metrics.items():
    if name not in dirs:
        continue
    # name is something like 'sptp_zll'
    # parse the factors from the name
    # e.g., system_prompt = name[5] (assuming fixed naming pattern)
    # temperature = name[6]
    # topp = name[7]
    
    system_prompt = name[5]  # adjust index based on your naming scheme
    temperature = name[6]
    topp = name[7]
    
    # For each metric column, append rows
    for column in df.columns:
        for val in df[column]:
            data_list.append({
                'MetricName': column,
                'Value': val,
                'SystemPrompt': system_prompt,
                'Temperature': temperature,
                'TopP': topp
            })

all_data = pd.DataFrame(data_list)
all_data

,MetricName,Value,SystemPrompt,Temperature,TopP
0,BLEU,0.000,z,l,l
1,BLEU,0.076,z,l,l
2,BLEU,0.061,z,l,l
3,BLEU,0.114,z,l,l
4,BLEU,0.272,z,l,l
...,...,...,...,...,...
986953,BERTScore_F1,0.858,c,h,h
986954,BERTScore_F1,0.887,c,h,h
986955,BERTScore_F1,0.834,c,h,h
986956,BERTScore_F1,0.855,c,h,h


In [351]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

for metric_name in all_data['MetricName'].unique():
    metric_df = all_data[all_data['MetricName'] == metric_name]
    # Fit the model: Value ~ SystemPrompt * Temperature * TopP
    model = ols('Value ~ C(SystemPrompt) * C(Temperature) * C(TopP)', data=metric_df).fit()
    anova_results = anova_lm(model, typ=2)
    print(f"ANOVA results for {metric_name}")
    print(anova_results)
    print("\n")

ANOVA results for BLEU
                                         sum_sq        df         F  PR(>F)
C(SystemPrompt)                         552.258     2.000 18290.909   0.000
C(Temperature)                            0.398     2.000    13.172   0.000
C(TopP)                                   0.818     2.000    27.090   0.000
C(SystemPrompt):C(Temperature)            0.931     4.000    15.422   0.000
C(SystemPrompt):C(TopP)                   1.715     4.000    28.404   0.000
C(Temperature):C(TopP)                    0.493     4.000     8.163   0.000
C(SystemPrompt):C(Temperature):C(TopP)    1.084     8.000     8.972   0.000
Residual                               1063.851 70470.000       NaN     NaN


ANOVA results for ROUGE-1_Precision
                                         sum_sq        df         F  PR(>F)
C(SystemPrompt)                         908.728     2.000 24791.899   0.000
C(Temperature)                            0.042     2.000     1.156   0.315
C(TopP)                    

In [353]:
all_data['Combination'] = (all_data['SystemPrompt'].astype(str) + "_" +
                           all_data['Temperature'].astype(str) + "_" +
                           all_data['TopP'].astype(str))

def run_posthoc_for_metric(metric_name):
    metric_df = all_data[all_data['MetricName'] == metric_name]
    
    model = ols('Value ~ C(Combination)', data=metric_df).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for {metric_name}:")
    print(anova_table)
    print("\n")

    tukey = pairwise_tukeyhsd(endog=metric_df['Value'],
                              groups=metric_df['Combination'],
                              alpha=0.05)
    print(f"Tukey HSD results for {metric_name}:")
    print(tukey.summary())
    print("\n")

    group_means = metric_df.groupby('Combination')['Value'].mean().sort_values(ascending=False)
    best_combination = group_means.index[0]
    best_mean = group_means.iloc[0]
    print(f"For {metric_name}, the highest mean is {best_mean:.4f} from {best_combination}.\n")

    tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], columns=tukey._results_table.data[0])
    significant_against_best = tukey_df[(tukey_df['group1'] == best_combination) | (tukey_df['group2'] == best_combination)]
    print(f"Comparisons involving the best combination ({best_combination}):")
    print(significant_against_best)
    print("="*80)


for metric_name in all_data['MetricName'].unique():
    run_posthoc_for_metric(metric_name)


ANOVA for BLEU:
                 sum_sq        df        F  PR(>F)
C(Combination)  557.696    26.000 1420.849   0.000
Residual       1063.851 70470.000      NaN     NaN


Tukey HSD results for BLEU:
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
 c_h_h  c_h_l   0.0476    0.0   0.035  0.0602   True
 c_h_h  c_h_m   0.0377    0.0  0.0252  0.0503   True
 c_h_h  c_l_h   0.0457    0.0  0.0332  0.0583   True
 c_h_h  c_l_l   0.0483    0.0  0.0357  0.0608   True
 c_h_h  c_l_m   0.0458    0.0  0.0333  0.0584   True
 c_h_h  c_m_h   0.0247    0.0  0.0121  0.0372   True
 c_h_h  c_m_l   0.0475    0.0  0.0349    0.06   True
 c_h_h  c_m_m   0.0414    0.0  0.0288   0.054   True
 c_h_h  f_h_h  -0.1603    0.0 -0.1729 -0.1477   True
 c_h_h  f_h_l  -0.1645    0.0 -0.1771 -0.1519   True
 c_h_h  f_h_m  -0.1648    0.0 -0.1774 -0.1522   True
 c_h_h  f_l_h  -0.1643    0.0 -0.1769 -0.1518   True
 c_h_h

# Placeholder Gang

In [212]:
metrics_values = {metric: [] for metric in metrics[list(metrics.keys())[0]].columns}

for k, v in metrics.items():
    for metric_name, metric_values in v.items():
        metrics_values[metric_name].extend(metric_values)

metrics_summary = []

for metric_name, all_values in metrics_values.items():
    min_val = min(all_values)
    mean_val = sum(all_values) / len(all_values)
    max_val = max(all_values)
    
    metrics_summary.append({'Metric': metric_name, 'Min': min_val, 'Mean': mean_val, 'Max': max_val})

metrics_summary_df = pd.DataFrame(metrics_summary)

metrics_summary_df.set_index('Metric', inplace=True)

metrics_summary_df

,Min,Mean,Max
Metric,,,
BLEU,0.000000,0.177187,1.000000
ROUGE-1_Precision,0.000000,0.263488,1.000000
ROUGE-1_Recall,0.000000,0.576460,1.000000
ROUGE-1_F1,0.000000,0.331782,1.000000
ROUGE-2_Precision,0.000000,0.195581,1.000000
ROUGE-2_Recall,0.000000,0.430168,1.000000
ROUGE-2_F1,0.000000,0.247461,1.000000
ROUGE-L_Precision,0.000000,0.234363,1.000000
ROUGE-L_Recall,0.000000,0.515112,1.000000
